In [ ]:
# Se leen las canastas de longitud variable como conjuntos de productos.

from pathlib import Path

data_path = Path("../data/groceries.csv")
transactions = []
for line in data_path.read_text(encoding="utf-8").splitlines():
    basket = frozenset(item.strip() for item in line.split(",") if item.strip())
    if basket:
        transactions.append(basket)
transactions[:5]

In [ ]:
# ¿Qué productos y tamaños de canasta dominan las transacciones del supermercado?

from collections import Counter
import pandas as pd

total_transactions = len(transactions)
item_counts = Counter(item for basket in transactions for item in basket)
top_items = pd.DataFrame([{"product": item, "basket_count": count, "support": count / total_transactions} for item, count in item_counts.items()]).sort_values("basket_count", ascending=False)
basket_sizes = pd.Series([len(basket) for basket in transactions], name="items_in_basket")
top_items.head(10), basket_sizes.value_counts().sort_index()

In [ ]:
# Se generan candidatos de Apriori únicamente a partir de productos con soporte suficiente.

from itertools import combinations

min_support = 0.006
singletons = {frozenset([item]): count / total_transactions for item, count in item_counts.items() if count / total_transactions >= min_support}
pair_candidates = {left | right for left, right in combinations(singletons, 2)}
pair_counts = Counter(candidate for basket in transactions for candidate in pair_candidates if candidate.issubset(basket))
frequent_pairs = {pair: count / total_transactions for pair, count in pair_counts.items() if count / total_transactions >= min_support}
len(singletons), len(frequent_pairs)

In [ ]:
# ¿Qué reglas tienen soporte, confianza y lift suficientes para recomendar productos?

rules = []
for pair, support in frequent_pairs.items():
    for antecedent_item in pair:
        antecedent = frozenset([antecedent_item])
        consequent = pair - antecedent
        confidence = support / singletons[antecedent]
        lift = confidence / singletons[consequent]
        if confidence >= 0.25 and lift >= 1.10:
            rules.append({"antecedent": antecedent_item, "consequent": next(iter(consequent)), "support": support, "confidence": confidence, "lift": lift})
rules = pd.DataFrame(rules).sort_values(["lift", "confidence"], ascending=False)
rules.head(10)

In [ ]:
# Se recomiendan productos cuando la canasta parcial contiene tropical fruit y yogurt.

selected_items = {"tropical fruit", "yogurt"}
recommendations = rules.loc[rules["antecedent"].isin(selected_items)].copy()
recommendations

In [ ]:
# Se conservan la exploración, las reglas y las recomendaciones para verificar el taller.

submission_dir = Path("../submission")
top_items.to_csv(submission_dir / "top_items.csv", index=False)
rules.to_csv(submission_dir / "association_rules.csv", index=False)
recommendations.to_csv(submission_dir / "recommendations.csv", index=False)